# Глава 33. Объяснимый ИИ (Explainable AI)

_Ноутбук сгенерирован из учебника «Машинное обучение для начинающих». Запускайте ячейки по порядку сверху вниз в Google Colab._

**Цель главы  🎯**

После этой главы ты сможешь объяснить, почему важна интерпретируемость моделей, применить SHAP для объяснения предсказаний и рассказать, почему модель приняла конкретное решение о конкретном объекте.

Модель отказала человеку в кредите. «Почему?» — спрашивает он. Ответ «так решила нейросеть» неприемлем — ни этически, ни юридически. Объяснимый ИИ (Explainable AI, XAI) — это набор методов, которые позволяют понять и объяснить, почему модель приняла то или иное решение. В твоём банковском контексте это особенно важно.

**Аналогия: чёрный ящик со стеклянной дверцей  💡**

Сложная модель (лес, нейросеть) — как чёрный ящик: данные входят, решение выходит, а что внутри — непонятно. Explainable AI приделывает к ящику стеклянную дверцу: теперь видно, какие признаки и как повлияли на конкретное решение. «Тебе отказали в кредите на 60% из-за короткой кредитной истории и на 30% из-за нестабильного дохода» — вот что даёт XAI вместо загадочного «нет».

## Два уровня объяснения

Есть два вопроса. Глобальный: какие признаки важны для модели вообще? На него отвечает feature importance из главы 9. Локальный: почему модель приняла именно ЭТО решение об ЭТОМ конкретном человеке? Вот на второй, самый острый вопрос и отвечает SHAP.

## SHAP: вклад каждого признака

**Аналогия: разложить счёт по участникам  💡**

Представь общий счёт в кафе, который надо честно разделить по тому, кто что заказал. SHAP делает то же с предсказанием: он берёт итоговое решение модели и честно раскладывает его на вклады каждого признака. Возраст добавил +0,2 к вероятности одобрения, низкий доход убавил −0,3, хорошая история добавила +0,15. Сумма вкладов даёт итоговое решение — и сразу видно, что перевесило.

In [ ]:
# === Подготовка: данные пингвинов ===
import pandas as pd
from sklearn.model_selection import train_test_split
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"
df = pd.read_csv(url).dropna(subset=["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g", "species"])
X = df[["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]]
y = df["species"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Запускать в Colab: !pip install shap -q
import shap
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(random_state=42).fit(X_train, y_train)

# Объясняем предсказания
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# Общая картина: какие признаки важны и как влияют
shap.summary_plot(shap_values, X_test)

График summary_plot показывает сразу все признаки, отсортированные по важности, и направление влияния каждого. Красные точки — высокое значение признака, синие — низкое; их положение показывает, толкают ли они решение вверх или вниз.

## Объяснение одного решения

Самое ценное — объяснить конкретное предсказание. Почему именно этому клиенту отказано?

In [ ]:
# Объясняем решение для одного конкретного объекта (например, клиента №0)
shap.force_plot(
    explainer.expected_value[1],
    shap_values[1][0],
    X_test.iloc[0]
)
# Покажет: какие признаки толкнули решение к "да", какие к "нет"

Такой график наглядно показывает: вот эти признаки склоняли модель к одобрению, а вот эти — к отказу, и что перевесило. Именно это можно показать человеку как объяснение решения.

## Локальный пример: объяснение отказа в кредите

Банк в Таджикистане обязан (и по-человечески должен) объяснять решения. С SHAP менеджер видит: «Клиенту отказано, главные причины — короткая кредитная история (вклад −0,4) и высокая долговая нагрузка (−0,25), при этом стабильная работа сыграла в плюс (+0,15)». Это можно объяснить клиенту, обжаловать, проверить на справедливость (глава 25). XAI превращает «машина так решила» в понятное, оспоримое и честное решение. Без него доверять модели важные решения о людях нельзя.

**Связь с этикой  🎯**

Explainable AI — родной брат главы 25 об этике. Если ты можешь объяснить решение модели, ты можешь проверить его на справедливость, найти скрытую предвзятость и защитить право человека понимать, почему о нём приняли то или иное решение. В ответственном ML объяснимость — не роскошь, а обязанность.

## Задания

Базовый уровень

Обучи случайный лес на пингвинах и построй shap.summary_plot. Какие признаки SHAP считает самыми влиятельными? Совпадает ли это с feature_importances_ из главы 9?

Со звёздочкой  ⭐

Выбери двух конкретных пингвинов разных видов по предсказанию модели и объясни каждое решение через SHAP. Какие признаки толкнули модель к одному виду, а какие — к другому? Напиши объяснение простым языком.

Вопросы для обсуждения  💬

Почему для решений о людях (кредит, приём на работу) объяснимость особенно важна?

Чем локальное объяснение (почему этот человек) ценнее глобального (что важно вообще)?

Как объяснимый ИИ помогает бороться с предвзятостью из главы 25?

**Главная мысль  🎯**

Ты научился не только строить модели, но и объяснять их решения — навык, который отличает ответственного специалиста. Чёрный ящик со стеклянной дверцей вызывает доверие, поддаётся проверке и уважает право человека на объяснение. В мире, где ИИ принимает всё больше решений, умение сказать «вот почему» становится критически важным.

**Проверь себя  ℹ️**

1. В чём разница между глобальным и локальным объяснением?

2. Что делает SHAP с предсказанием модели?

3. Почему объяснимость связана с этикой и справедливостью?

# Что дальше: твои следующие шаги

Эта книга — не конец, а стартовая площадка. Ты освоил фундамент, на котором можно строить сколько угодно. Вот куда двигаться, чтобы расти дальше.

### Соревнования

Kaggle (kaggle.com) — площадка с реальными задачами и соревнованиями со всего мира. Начни с раздела «Getting Started» с любой задачи для начинающих — там понятные данные и много разобранных решений. Соревнования — лучший способ учиться быстро: ты видишь чужие решения, сравниваешь подходы и растёшь на реальных данных.

### Углубление теории

Когда захочешь понять, что под капотом, глубже: канал 3Blue1Brown на YouTube (потрясающие визуальные объяснения нейросетей), StatQuest (простым языком о статистике и ML), Kaggle Learn (бесплатные короткие курсы с практикой). Теория станет намного понятнее теперь, когда за плечами есть практика.

### Свои проекты

Самое ценное — продолжать делать. Придумай проект на тему, которая тебя по-настоящему увлекает: любимая игра, музыка, спорт, хобби. Выкладывай проекты на GitHub и Hugging Face — так собирается портфолио, которое говорит за тебя лучше любого резюме. Три-четыре живых проекта впечатляют работодателя или приёмную комиссию сильнее любых сертификатов.

### Хакатоны и сообщество

Ищи хакатоны для школьников и студентов, вступай в сообщества по ИИ. Учиться вместе с другими — быстрее, веселее и полезнее, чем в одиночку. Там же находятся будущие коллеги и друзья по интересам.

**Финальное слово  🎯**

Ты доказал себе главное: искусственный интеллект — не магия для избранных, а инструмент, которым ты умеешь пользоваться. За эти три месяца ты создавал настоящие технологические продукты своими руками — от анализа данных до работающих приложений в интернете. Не останавливайся: впереди самое интересное, а ты уже знаешь, что тебе это по силам. Удачи!

# Глоссарий

Краткий словарь терминов, которые встречались в книге. Возвращайся сюда, когда забудешь, что значит слово.

Attention (внимание) — Механизм, позволяющий модели решать, на какие слова смотреть.

AutoML — Инструменты, автоматически подбирающие модель и её параметры.

Baseline — Простейшая модель как точка отсчёта, с которой сравнивают настоящую модель.

CNN (свёрточная сеть) — Нейросеть для изображений, которая находит узоры фильтрами-рамками.

Confusion matrix — Таблица, показывающая типы ошибок классификатора.

Data leakage (утечка данных) — Ситуация, когда модель случайно «подсматривает» информацию, недоступную в реальности.

Deploy (деплой) — Выкладывание приложения в интернет по постоянному адресу.

EDA — Разведочный анализ данных — изучение данных через графики и статистику.

Embedding (эмбеддинг) — Запись смысла слова набором чисел; похожие по смыслу слова оказываются рядом.

F1 — Метрика, балансирующая precision и recall.

Feature (признак) — Столбец-характеристика в данных, по которому модель делает предсказание.

Gradio — Библиотека для быстрого создания веб-интерфейса к модели.

KNN — Алгоритм «ближайших соседей»: объект относят к тому классу, что у соседей.

LLM — Большая языковая модель — предсказывает следующее слово, стоит за чат-ботами.

MAE / MSE — Метрики регрессии: средняя ошибка и средний квадрат ошибки.

Overfitting (переобучение) — Модель зазубрила обучающие данные и плохо работает на новых.

Pipeline — Конвейер, объединяющий предобработку и модель; защищает от утечки данных.

Precision / Recall — Метрики классификации: точность попаданий и полнота найденного.

Prompt (промпт) — Текстовая инструкция, которую даёшь языковой модели.

RAG — Приём, дающий LLM «шпаргалку» из документа, чтобы уменьшить галлюцинации.

Random Forest (случайный лес) — Ансамбль из многих деревьев решений, голосующих вместе.

Regression (регрессия) — Задача предсказания числа (в отличие от категории).

Token (токен) — Кусочек текста, на которые LLM режет входные данные.

Train / Test — Разделение данных: на train модель учится, на test честно проверяется.

Transfer learning — Дообучение готовой модели под свою задачу вместо обучения с нуля.

Аугментация — Искусственное расширение датасета поворотами, отражениями и т. п.

Эпоха (epoch) — Один полный проход обучения по всем данным.

ПРИЛОЖЕНИЯ

Справочная часть

_Шпаргалки, каталоги, чек-листы и ответы — то, к чему возвращаются снова и снова._

Эта часть книги устроена иначе, чем главы: её не читают подряд, к ней обращаются по мере надобности. Забыл, как отфильтровать таблицу, — открой шпаргалку по Pandas. Ищешь тему проекта — загляни в банк идей. Не помнишь термин — словарь. Держи книгу под рукой во время практики: приложения сэкономят тебе десятки поисков в интернете.